Risk Aware RL environment

In [1]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import random
import heapq
import itertools
import time
from enum import IntEnum

class Actions(IntEnum):
    HOLD = 0
    BUY = 1
    SELL = 2

# Dimensions: 5 Bids + 5 Asks + Inv + Cash + Spread + Vol = 14
OBSERVATION_DIMS = 14

# Reward Hyperparameters
INVENTORY_RISK_COEFF = 0.1   # Penalty per unit of inventory held
DOWNSIDE_PENALTY_MULT = 10.0 # Heavy penalty for drawdowns
TRANSACTION_COST = 1.0       # Cost per trade to prevent churning

# Env Settings
INITIAL_CASH = 100_000.0
MAX_STEPS_PER_EPISODE = 2000
MAX_INVENTORY_LIMIT = 100
MAX_CASH_LIMIT = 1_000_000


class AdvancedOrderBook:
    """
    The Matching Engine (The "World" logic).
    Handles order matching, book management, and trade execution.
    """
    def __init__(self):
        self.bids = []  # Max-Heap (negative prices)
        self.asks = []  # Min-Heap (positive prices)
        self.trades = []
        self.order_id_counter = itertools.count() 

    def submit_order(self, side, qty, price=None, order_type='limit'):
        if order_type == 'market':
            limit_price = float('inf') if side == 'buy' else 0
        else:
            limit_price = price

        remaining_qty = self.match(side, qty, limit_price)

        if remaining_qty > 0 and order_type == 'limit':
            entry_id = next(self.order_id_counter)
            if side == 'buy':
                # Python has only min-heap, so we negate bid price to simulate max-heap
                heapq.heappush(self.bids, [-limit_price, entry_id, remaining_qty])
            else:
                heapq.heappush(self.asks, [limit_price, entry_id, remaining_qty])
            
        return remaining_qty

    def match(self, side, qty, limit_price):
        remaining_qty = qty
        while remaining_qty > 0:
            if side == 'buy':
                if not self.asks: break
                best_price = self.asks[0][0]
                if limit_price < best_price: break
                best_order = self.asks[0]
            else:
                if not self.bids: break
                best_price = -self.bids[0][0] 
                if limit_price > best_price: break
                best_order = self.bids[0]

            # EXECUTION
            trade_qty = min(remaining_qty, best_order[2])
            exec_price = best_order[0] if side == 'buy' else -best_order[0]

            self.trades.append({
                'price': exec_price,
                'qty': trade_qty,
                'timestamp': time.time(),
                'side': side,
                'aggressor': 'market' if limit_price == float('inf') else 'limit'
            })

            remaining_qty -= trade_qty
            best_order[2] -= trade_qty

            # Cleanup
            if best_order[2] == 0:
                if side == 'buy': heapq.heappop(self.asks)
                else: heapq.heappop(self.bids)
                    
        return remaining_qty

# 3. RL ENVIRONMENT (With Enhanced Reward)

class TradingEnv(gym.Env):
    metadata = {'render_modes': ['human']}

    def __init__(self):
        super(TradingEnv, self).__init__()
        
        # Action Space: Hold, Buy, Sell
        self.action_space = spaces.Discrete(len(Actions))
        
        # Observation Space: Normalized market data
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, 
            shape=(OBSERVATION_DIMS,), dtype=np.float32
        )
        
        # Internal State
        self.engine = None
        self.inventory = 0
        self.cash = INITIAL_CASH
        self.portfolio_value = INITIAL_CASH
        self.max_portfolio_value = INITIAL_CASH # Track peak for drawdown calc
        self.current_step = 0
        self.mid_price_history = [] 

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        self.engine = AdvancedOrderBook()
        
        # Seed the Market with dummy orders
        start_price = 100.0
        for i in range(1, 6):
            self.engine.submit_order('buy', 10, start_price - i*0.5, 'limit')
            self.engine.submit_order('sell', 10, start_price + i*0.5, 'limit')

        self.inventory = 0
        self.cash = INITIAL_CASH
        self.portfolio_value = INITIAL_CASH
        self.max_portfolio_value = INITIAL_CASH
        self.current_step = 0
        self.mid_price_history = [start_price] * 10 
        
        return self._get_observation(), {}

    def step(self, action):
        prev_portfolio_value = self.portfolio_value
        
        # 1. Execute Action
        self._execute_action(action)
        
        # 2. Simulate Market Dynamics
        self._simulate_background_market()
        
        # 3. Update Portfolio Value (Mark-to-Market)
        current_mid = self._get_mid_price()
        self.portfolio_value = self.cash + (self.inventory * current_mid)
        
        # Track Peak for Drawdown
        if self.portfolio_value > self.max_portfolio_value:
            self.max_portfolio_value = self.portfolio_value
            
        # 4. REWARD ENGINEERING 
        
        # A. Incremental PnL (Making money is good)
        delta_pnl = self.portfolio_value - prev_portfolio_value
        
        # B. Inventory Risk Penalty (Holding inventory is dangerous)
        # We penalize the absolute inventory size.
        inventory_penalty = INVENTORY_RISK_COEFF * abs(self.inventory)
        
        # C. Drawdown Penalty (Losing capital is very bad)
        # If current value is 2% below peak, apply harsh penalty
        drawdown_pct = (self.max_portfolio_value - self.portfolio_value) / self.max_portfolio_value
        drawdown_penalty = 0
        if drawdown_pct > 0.02: 
            drawdown_penalty = drawdown_pct * DOWNSIDE_PENALTY_MULT
            
        # D. Transaction Cost (Already applied to cash, but reinforced in reward)
        # If we traded, we pay a "cost of attention"
        trade_cost = 0
        if action != Actions.HOLD:
            trade_cost = TRANSACTION_COST

        # TOTAL REWARD
        reward = delta_pnl - inventory_penalty - drawdown_penalty - trade_cost
        
        # ==========================================
        
        self.current_step += 1
        
        # 5. Check Termination
        # Stop if bankrupt OR max steps reached
        terminated = self.portfolio_value <= 0 
        truncated = self.current_step >= MAX_STEPS_PER_EPISODE
        
        observation = self._get_observation()
        info = {
            'portfolio_value': self.portfolio_value, 
            'inventory': self.inventory,
            'drawdown': drawdown_pct,
            'reward_components': {
                'pnl': delta_pnl,
                'inv_penalty': inventory_penalty,
                'dd_penalty': drawdown_penalty
            }
        }
        
        return observation, reward, terminated, truncated, info

    def _execute_action(self, action):
        if action == Actions.HOLD: return
        qty = 1
        if action == Actions.BUY:
            if self.cash > 0: 
                rem = self.engine.submit_order('buy', qty, order_type='market')
                if rem == 0: 
                    fill_price = self.engine.trades[-1]['price']
                    self.inventory += qty
                    self.cash -= fill_price
        elif action == Actions.SELL:
            rem = self.engine.submit_order('sell', qty, order_type='market')
            if rem == 0: 
                fill_price = self.engine.trades[-1]['price']
                self.inventory -= qty
                self.cash += abs(fill_price)

    def _simulate_background_market(self):
        # Random Walk Logic to move prices
        mid = self._get_mid_price()
        shock = np.random.normal(0, 0.5)
        new_fair = mid + shock
        self.engine.submit_order('buy', 10, round(new_fair - 0.5, 2), 'limit')
        self.engine.submit_order('sell', 10, round(new_fair + 0.5, 2), 'limit')

    def _get_mid_price(self):
        best_bid = -self.engine.bids[0][0] if self.engine.bids else 100.0
        best_ask = self.engine.asks[0][0] if self.engine.asks else 100.0
        mid = (best_bid + best_ask) / 2
        
        self.mid_price_history.append(mid)
        if len(self.mid_price_history) > 20: self.mid_price_history.pop(0)
        return mid

    def _get_observation(self):
        mid = self._get_mid_price()
        
        # Safe access to top 5 levels
        bids = [-x[0] for x in heapq.nsmallest(5, self.engine.bids)] if self.engine.bids else []
        asks = [x[0] for x in heapq.nsmallest(5, self.engine.asks)] if self.engine.asks else []
        
        while len(bids) < 5: bids.append(mid)
        while len(asks) < 5: asks.append(mid)
        
        # Normalize
        norm_bids = [(p - mid)/mid for p in bids]
        norm_asks = [(p - mid)/mid for p in asks]
        norm_inv = self.inventory / MAX_INVENTORY_LIMIT
        norm_cash = self.cash / MAX_CASH_LIMIT
        spread = (asks[0] - bids[0]) / mid
        vol = np.std(self.mid_price_history) / mid if len(self.mid_price_history) > 1 else 0
        
        obs = np.array(norm_bids + norm_asks + [norm_inv, norm_cash, spread, vol], dtype=np.float32)
        return obs

# TEST HARNESS (Validate Reward Logic)
if __name__ == "__main__":
    print("--- Testing Risk-Aware Reward Function ---")
    env = TradingEnv()
    env.reset()
    
    # 1. Test Inventory Penalty
    print("\nTest 1: Accumulating Inventory (Should see negative reward)")
    for _ in range(5):
        # Force BUY actions
        obs, reward, term, trunc, info = env.step(Actions.BUY)
        comps = info['reward_components']
        print(f"Action: BUY | Inv: {info['inventory']} | Reward: {reward:.4f} | InvPenalty: {comps['inv_penalty']:.4f}")

    # 2. Test Drawdown Penalty (Simulated Crash)
    print("\nTest 2: Simulating Market Crash (Should see massive penalty)")
    # Artificially crash the market by submitting a low sell order
    env.engine.submit_order('sell', 100, 50.0, 'limit') # Huge sell wall low down
    env.engine.submit_order('sell', 100, 50.0, 'market') # Crash the price
    
    # Agent Holds during crash
    obs, reward, term, trunc, info = env.step(Actions.HOLD)
    comps = info['reward_components']
    print(f"Action: HOLD | Portfolio: {info['portfolio_value']:.2f} | Drawdown: {info['drawdown']:.2%} | DDPenalty: {comps['dd_penalty']:.4f}")
    
    print("\nTest Complete.")

--- Testing Risk-Aware Reward Function ---

Test 1: Accumulating Inventory (Should see negative reward)
Action: BUY | Inv: 1 | Reward: -1.8500 | InvPenalty: 0.1000
Action: BUY | Inv: 2 | Reward: -1.2000 | InvPenalty: 0.2000
Action: BUY | Inv: 3 | Reward: -2.0500 | InvPenalty: 0.3000
Action: BUY | Inv: 4 | Reward: -0.8100 | InvPenalty: 0.4000
Action: BUY | Inv: 5 | Reward: -1.5400 | InvPenalty: 0.5000

Test 2: Simulating Market Crash (Should see massive penalty)
Action: HOLD | Portfolio: 99934.85 | Drawdown: 0.07% | DDPenalty: 0.0000

Test Complete.
